# Delaunay-filtered proximity graphs

Compare all-pairs Gabriel/RNG reference methods with versions that first restrict candidate edges to a Delaunay triangulation. The filter shown here is still O(n²): it scans all sites for each sparse candidate edge.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'implementations').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'implementations/python/proximity-graphs'))
from proximity_graphs import (gabriel_graph, relative_neighborhood_graph,
    delaunay_filtered_gabriel_graph, delaunay_filtered_rng)
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import Delaunay

## 1. Compare outputs on a general-position point set

The chosen points avoid obvious cocircular ties. The candidate filter should agree with the all-pairs reference for these examples.

In [ ]:
points=[(0,0),(4,0),(5,3),(2,5),(-1,3),(1,2)]
print('Gabriel:', gabriel_graph(points))
print('Delaunay-filtered Gabriel:', delaunay_filtered_gabriel_graph(points))
print('RNG:', relative_neighborhood_graph(points))
print('Delaunay-filtered RNG:', delaunay_filtered_rng(points))
assert delaunay_filtered_gabriel_graph(points)==gabriel_graph(points)
assert delaunay_filtered_rng(points)==relative_neighborhood_graph(points)

In [ ]:
tri=Delaunay(np.asarray(points,dtype=float))
candidates=set()
for a,b,c in tri.simplices:
    for u,v in ((a,b),(b,c),(c,a)): candidates.add(tuple(sorted((int(u),int(v)))))
print('all possible pairs:', len(points)*(len(points)-1)//2)
print('Delaunay candidate edges:', len(candidates))
print('retained Gabriel edges:', len(delaunay_filtered_gabriel_graph(points)))
print('retained RNG edges:', len(delaunay_filtered_rng(points)))

## 2. Visualise the candidate graph

A planar triangulation has only linearly many edges. The diagram separates candidate edges from retained Gabriel and RNG edges.

In [ ]:
def draw_graph(ax, edges, title):
    ax.scatter([p[0] for p in points],[p[1] for p in points],zorder=3)
    for i,(x,y) in enumerate(points): ax.annotate(str(i),(x,y),xytext=(4,4),textcoords='offset points')
    for i,j in edges:
        ax.plot([points[i][0],points[j][0]],[points[i][1],points[j][1]],alpha=.75)
    ax.set_aspect('equal',adjustable='box'); ax.grid(True); ax.set_title(title)
fig,axes=plt.subplots(1,3,figsize=(14,4))
draw_graph(axes[0],sorted(candidates),'Delaunay candidates')
draw_graph(axes[1],delaunay_filtered_gabriel_graph(points),'Gabriel')
draw_graph(axes[2],delaunay_filtered_rng(points),'RNG')
plt.tight_layout(); plt.show()

## 3. Understand the complexity gap

The all-pairs reference checks O(n²) candidate pairs against O(n) possible blockers: O(n³). The Delaunay-filtered version checks O(n) candidate edges against O(n) sites: O(n²). This is an improvement, but it is not the specialised linear-time RNG post-processing algorithm.

## 4. Degeneracy experiment

Try a square. With the open-diameter-disk Gabriel rule, both diagonals qualify because the opposite corners lie on the boundary. A triangulation chooses one diagonal, so the filtered method can omit the other. This is a deliberate reminder that a Delaunay candidate filter needs an explicit degeneracy policy.

In [ ]:
square=[(0,0),(1,0),(1,1),(0,1)]
print('all-pairs Gabriel:',gabriel_graph(square))
print('Delaunay-filtered Gabriel:',delaunay_filtered_gabriel_graph(square))